# Measuring the saving [Step 05.04 - Real numbers, honestly reported]

> **MLCourse - Agentic AI - Agent Patterns**

Everything so far has been mechanism. This notebook answers the only question that
matters to whoever approves the work: **how much does routing actually save, on a
workload that looks like yours?**

We run the same 24-request workload twice:

1. **Baseline** - every request goes to the model. This is what you have today.
2. **Routed** - the router handles what it can; the rest goes to the same model.

Then we compare tokens, dollars and wall-clock latency, and we count the mistakes
routing introduced. A saving with an unstated error cost is not a result.

### Key takeaways

- The saving is entirely determined by **what fraction of your traffic is
  repetitive**. Routing a workload of genuinely novel questions saves nothing.
- Report the saving *and* the misroutes. One number without the other is marketing.
- The router's own cost is real but tiny, and it is CPU, not tokens.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


### The embedding model


In [ ]:
# `all-MiniLM-L6-v2` is 22M parameters, runs on CPU, and encodes a short sentence
# in single-digit milliseconds. That speed is the whole point: routing has to be
# cheap enough that it is obviously worth doing before the expensive call.

from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")


def embed(texts):
    """Encode a list of strings into L2-normalised vectors.

    Normalising means the dot product IS the cosine similarity, so every score
    below lives in [-1, 1] and is directly comparable.
    """
    return encoder.encode(list(texts), normalize_embeddings=True)


v = embed(["hello there", "hi!", "what is the capital of France"])
print("vector shape :", v.shape)
print("hello/hi     : %.3f" % float(v[0] @ v[1]))
print("hello/capital: %.3f" % float(v[0] @ v[2]))


### The route set


In [ ]:
# A ROUTE is: a name, a handful of EXAMPLE UTTERANCES, and a handler.
# Nothing more. There is no training step and no classifier to fit.

ROUTES = {
    "greeting": [
        "hi there",
        "hello",
        "hey, good morning",
        "yo",
        "good evening",
    ],
    "shipping_faq": [
        "how long does delivery take",
        "when will my parcel arrive",
        "do you ship internationally",
        "what are your shipping costs",
        "how fast is standard delivery",
    ],
    "order_status": [
        "where is order 1042",
        "track my order 88",
        "what happened to order number 7",
        "status of order 1042 please",
        "has order 88 shipped yet",
    ],
    "arithmetic": [
        "what is 18 times 24",
        "compute 145 plus 92",
        "calculate 900 divided by 12",
        "how much is 37 minus 19",
        "multiply 13 by 7",
    ],
}

ROUTE_NAMES = list(ROUTES)
print("%d routes, %d example utterances total"
      % (len(ROUTES), sum(len(v) for v in ROUTES.values())))


### 1. The workload

24 requests with a mix that is deliberately realistic for a shop's support inbox:
lots of greetings and repeated FAQs, some order lookups, a few sums, and a tail of
genuinely open questions that *must* reach the model.

The `gold` field records what should happen, so we can count mistakes rather than
assume there were none.

In [4]:
# (request, gold route or None = must reach the model)
WORKLOAD = [
    ("hi there",                                    "greeting"),
    ("good morning",                                "greeting"),
    ("hello!",                                      "greeting"),
    ("hey",                                         "greeting"),
    ("evening",                                     "greeting"),
    ("how long does delivery take",                 "shipping_faq"),
    ("what are your shipping costs",                "shipping_faq"),
    ("do you deliver to Portugal",                  "shipping_faq"),
    ("is next day delivery an option",              "shipping_faq"),
    ("how much is postage to Italy",                "shipping_faq"),
    ("what is the delivery time to Germany",        "shipping_faq"),
    ("where is order 1042",                         "order_status"),
    ("track order 88",                              "order_status"),
    ("status of order 7",                           "order_status"),
    ("has order 1042 shipped",                      "order_status"),
    ("what is 18 times 24",                         "arithmetic"),
    ("add 210 and 66",                              "arithmetic"),
    ("divide 480 by 16",                            "arithmetic"),
    ("what is your returns policy for damaged items", None),
    ("can I change the delivery address after ordering", None),
    ("do you offer gift wrapping and how does it work", None),
    ("my parcel arrived with a missing item, what now", None),
    ("who wrote Pride and Prejudice",                None),
    ("explain the difference between TCP and UDP",   None),
]
n_route = sum(1 for _, g in WORKLOAD if g is not None)
print("%d requests: %d routable, %d must reach the model (%.0f%% routable)"
      % (len(WORKLOAD), n_route, len(WORKLOAD) - n_route, 100 * n_route / len(WORKLOAD)))

24 requests: 18 routable, 6 must reach the model (75% routable)


> **Be honest about this mix.** 75% routable is a support inbox. A research
> assistant's traffic might be 5% routable, and routing would then save almost
> nothing while adding a failure mode. Measure your own mix before you build this.

### the same handlers and router as notebook 03, condensed


In [ ]:
import re

ORDER_RE = re.compile(r"order\s*(?:number\s*|#\s*|no\.?\s*)?(\d+)", re.I)
ORDERS = {"7": "delivered 12 March", "88": "in transit, arriving Thursday",
          "1042": "packed, leaves the warehouse tonight"}
THRESHOLD = 0.45

ROUTE_VECTORS = {n: embed(ex) for n, ex in ROUTES.items()}


def top_score(text):
    q = embed([text])[0]
    s = {n: float((v @ q).max()) for n, v in ROUTE_VECTORS.items()}
    b = max(s, key=s.get)
    return b, s[b]


def h_greeting(t):
    return "Hello! How can I help with your order today?"


def h_faq(t):
    return ("Standard delivery is 3-5 working days within the EU and 7-10 days "
            "internationally. Shipping is free over EUR 40.")


def h_order(t):
    m = ORDER_RE.search(t)
    return None if not m else "Order %s: %s." % (m.group(1), ORDERS.get(m.group(1), "no record"))


def h_math(t):
    nums = [int(n) for n in re.findall(r"-?\d+", t)]
    if len(nums) != 2:
        return None
    a, b = nums
    low = t.lower()
    if any(w in low for w in ("times", "multiply")):
        return "%d x %d = %d" % (a, b, a * b)
    if any(w in low for w in ("plus", "add")):
        return "%d + %d = %d" % (a, b, a + b)
    if "divid" in low:
        return "%d / %d = %.4g" % (a, b, a / b)
    if any(w in low for w in ("minus", "subtract")):
        return "%d - %d = %d" % (a, b, a - b)
    return None


HANDLERS = {"greeting": h_greeting, "shipping_faq": h_faq,
            "order_status": h_order, "arithmetic": h_math}

llm = make_llm(temperature=0.0, max_tokens=160)
SYSTEM = ("You are a support assistant for an online shop. Answer in at most two "
          "sentences. If the question is outside the shop's domain, say so plainly.")


def call_model(text):
    msg = safe_invoke(llm, [("system", SYSTEM), ("user", text)])
    u = msg.usage_metadata or {}
    return msg.content.strip(), u.get("input_tokens", 0), u.get("output_tokens", 0)


print("handlers and model ready")


### 2. Baseline: call the model for everything

24 small calls, paced for the free tier. We record tokens per request so the
comparison later is exact rather than estimated.

In [6]:
base_in = base_out = 0
base_t0 = time.time()
baseline_answers = []
for i, (text, gold) in enumerate(WORKLOAD, 1):
    ans, ti, to = call_model(text)
    base_in += ti
    base_out += to
    baseline_answers.append(ans)
    print("%2d/%d  %-48s  %3d+%3d tok" % (i, len(WORKLOAD), text[:48], ti, to))
base_seconds = time.time() - base_t0 - PACE * len(WORKLOAD)   # remove our own sleeps

print()
print("BASELINE: %d calls, %d in + %d out = %d tokens, %.2f s of API time, $%.6f"
      % (len(WORKLOAD), base_in, base_out, base_in + base_out, base_seconds,
         usd(base_in, base_out)))

 1/24  hi there                                           50+ 14 tok


 2/24  good morning                                       50+ 14 tok


 3/24  hello!                                             50+ 14 tok


 4/24  hey                                                49+ 15 tok


 5/24  evening                                            50+ 16 tok


...

20/24  can I change the delivery address after ordering   56+ 35 tok


21/24  do you offer gift wrapping and how does it work    58+ 41 tok


22/24  my parcel arrived with a missing item, what now    58+ 41 tok


23/24  who wrote Pride and Prejudice                      55+ 14 tok


24/24  explain the difference between TCP and UDP         55+ 24 tok

BASELINE: 24 calls, 1298 in + 642 out = 1940 tokens, 8.65 s of API time, $0.000755


### 3. Routed: the router first, the model only when needed


In [7]:
route_in = route_out = 0
handled = 0
misroutes = []
router_seconds = 0.0
route_t0 = time.time()
routed_answers = []

for text, gold in WORKLOAD:
    r0 = time.time()
    best, sc = top_score(text)
    taken = best if sc >= THRESHOLD else None
    result = HANDLERS[taken](text) if taken else None
    router_seconds += time.time() - r0

    if result is not None:
        handled += 1
        routed_answers.append((text, taken, result))
        if gold != taken:
            misroutes.append((text, gold, taken, sc))
    else:
        ans, ti, to = call_model(text)
        route_in += ti
        route_out += to
        routed_answers.append((text, "model", ans))
        if gold is not None:
            pass          # a routable request that fell back: costs money, not correctness

route_seconds = time.time() - route_t0 - PACE * (len(WORKLOAD) - handled)

print("ROUTED: %d/%d handled locally (%.0f%% coverage)"
      % (handled, len(WORKLOAD), 100 * handled / len(WORKLOAD)))
print("        %d model calls, %d in + %d out = %d tokens, $%.6f"
      % (len(WORKLOAD) - handled, route_in, route_out, route_in + route_out,
         usd(route_in, route_out)))
print("        router CPU time total: %.3f s (%.1f ms per request)"
      % (router_seconds, router_seconds / len(WORKLOAD) * 1000))

ROUTED: 19/24 handled locally (79% coverage)
        5 model calls, 280 in + 150 out = 430 tokens, $0.000170
        router CPU time total: 0.179 s (7.4 ms per request)


### 4. The result


In [8]:
tok_base = base_in + base_out
tok_route = route_in + route_out
cost_base = usd(base_in, base_out)
cost_route = usd(route_in, route_out)

print("=" * 66)
print("%-26s %14s %14s %9s" % ("", "always-model", "routed", "saving"))
print("-" * 66)
print("%-26s %14d %14d %8.1f%%" % ("model calls", len(WORKLOAD), len(WORKLOAD) - handled,
                                   100 * handled / len(WORKLOAD)))
print("%-26s %14d %14d %8.1f%%" % ("input tokens", base_in, route_in,
                                   100 * (1 - route_in / base_in)))
print("%-26s %14d %14d %8.1f%%" % ("output tokens", base_out, route_out,
                                   100 * (1 - route_out / base_out)))
print("%-26s %14d %14d %8.1f%%" % ("total tokens", tok_base, tok_route,
                                   100 * (1 - tok_route / tok_base)))
print("%-26s %13.6f %13.6f %8.1f%%" % ("cost (USD)", cost_base, cost_route,
                                       100 * (1 - cost_route / cost_base)))
print("%-26s %13.2f %13.2f %8.1f%%" % ("API seconds", base_seconds, route_seconds,
                                       100 * (1 - route_seconds / base_seconds)))
print("-" * 66)
print("%-26s %14s %14d" % ("misroutes (wrong handler)", "n/a", len(misroutes)))
print("=" * 66)
print()
print("Per 1,000,000 requests of this mix:")
print("  always-model : $%.2f" % (cost_base / len(WORKLOAD) * 1e6))
print("  routed       : $%.2f" % (cost_route / len(WORKLOAD) * 1e6))
print("  saved        : $%.2f" % ((cost_base - cost_route) / len(WORKLOAD) * 1e6))

                             always-model         routed    saving
------------------------------------------------------------------
model calls                            24              5     79.2%
input tokens                         1298            280     78.4%
output tokens                         642            150     76.6%
total tokens                         1940            430     77.8%
cost (USD)                      0.000755      0.000170     77.5%
API seconds                         8.65          1.65     80.9%
------------------------------------------------------------------
misroutes (wrong handler)             n/a              1

Per 1,000,000 requests of this mix:
  always-model : $31.47
  routed       : $7.07
  saved        : $24.40


In [9]:
print("Misroutes (the cost side of the ledger):")
if not misroutes:
    print("  none on this workload - but note the workload is 24 requests, which")
    print("  cannot detect a misroute rate below about 4%.")
else:
    for text, gold, taken, sc in misroutes:
        print("  %-46s gold=%-13s taken=%-13s score=%.3f" % (text[:46], gold, taken, sc))

print()
print("Requests that were routable but still reached the model (pure lost saving):")
lost = [(t, g) for (t, g), (_, taken, _) in zip(WORKLOAD, routed_answers)
        if g is not None and taken == "model"]
for t, g in lost:
    print("  %-46s gold=%s" % (t[:46], g))
if not lost:
    print("  none")

Misroutes (the cost side of the ledger):
  my parcel arrived with a missing item, what no gold=None          taken=shipping_faq  score=0.576

Requests that were routable but still reached the model (pure lost saving):
  none


### 5. Reading the result honestly

Three caveats that belong in any write-up of a number like this:

1. **The saving is a property of the traffic mix, not of the technique.** We chose a
   75%-routable workload. Halve that and the saving roughly halves.

2. **The routed answers are not the same answers.** The canned FAQ string is shorter
   and less conversational than the model's. For a support bot that is often an
   improvement - it is consistent and approved. For other products it is a
   regression. Routing trades variety for cost; decide which you want.

3. **24 requests cannot measure a small error rate.** Zero misroutes here means
   "below roughly 4%", not "zero". Before shipping, run several hundred logged real
   requests through the router offline and grade the disagreements by hand.

### When routing is not worth it

- Traffic that is mostly novel (research, coding, open-ended chat) - coverage will
  be a few percent and you have added a failure mode for nothing.
- Very small volumes. Saving 60% of $3/month is not worth the route set you have
  to maintain.
- Cases where answer variety is the product.

### Where to go next

- `06_multi_agent_debate` - the opposite trade: spend *more* to try to be righter.
- `../../05_production_security/03_caching_strategies` - the other big cost lever;
  caching and routing compose well (route first, then cache the fallback).
- `../../03_rag_advanced/11_reranking` - the same "cheap stage in front of an
  expensive stage" shape, applied to retrieval.